# 10 - Expertos híbridos: intensidad, patrón, topología y grafos

Este nodo investiga la complementariedad observada entre los nodos 06 y 07. No crea una CNN 4D artificial. Construye dos vistas del **mismo crop físico**:

- `volume_intensity`: reconstruye captación registrada truncada como `volume_intensity01 × foreground_p99_registered` y la escala sólo con el train de cada fold;
- `volume_selfnorm`: proyecta esos mismos vóxeles mediante $\sqrt{d}x/\lVert x
Vert_2$.

La intensidad entra en la CNN 3D de dos maneras: como valores voxel a voxel de la primera rama y como cuatro escalares explícitos (`L2`, media, p90 y soporte positivo). La segunda rama aprende el patrón sin escala. La concatenación tiene tres compuertas sigmoid para impedir que una vista anule numéricamente a la otra.

Además se comparan un experto regional de 930 variables, topología multiumbral, tres baselines tabulares del mismo grafo, una GCN pequeña, Diffusion Maps con Nyström y una mezcla de subtipos. La etapa final genera OOF de cinco folds, cross-calibra cada experto, calcula una media preespecificada y optimiza un stacking regularizado exploratorio.


In [1]:
from __future__ import annotations

import os
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modeling.cnn.io import RunLock
from modeling.node10_hybrid import (
    ExperimentConfig,
    prepare_experiment,
    run_final_stage,
    run_search_stage,
)

RUN_ID = os.environ.get("DAT_NODE10_RUN_ID", "node10_hybrid_v1")
NODE4_PROFILE = os.environ.get("DAT_NODE10_NODE4_PROFILE", "v3")
COMPLETED_TRIALS = max(10, int(os.environ.get("DAT_NODE10_COMPLETED_TRIALS", "10")))
STACK_TRIALS = max(10, int(os.environ.get("DAT_NODE10_STACK_TRIALS", "10")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPERIMENT = ExperimentConfig(run_id=RUN_ID)
EXPERIMENT = replace(
    EXPERIMENT,
    data=replace(EXPERIMENT.data, node4_profile=NODE4_PROFILE),
    search=replace(
        EXPERIMENT.search,
        completed_trials_per_model=COMPLETED_TRIALS,
        stack_completed_trials=STACK_TRIALS,
    ),
)
RUN_DIR = PROJECT_ROOT / "outputs" / "private_eda" / "node10_runs" / RUN_ID

display(Markdown(
    f"**Run:** `{RUN_ID}` · **device:** `{DEVICE}` · **familias base:** "
    f"`{len(EXPERIMENT.model_families)}` · **trials COMPLETE mínimos por familia:** "
    f"`{EXPERIMENT.search.effective_completed_trials}`"
))


**Run:** `node10_hybrid_v1` · **device:** `cuda` · **familias base:** `9` · **trials COMPLETE mínimos por familia:** `10`

## 1. Preparación reanudable y auditoría de contrato

Se reutilizan los 1362 pacientes únicos del nodo 04. El caché se persiste caso a caso; si se interrumpe, continúa con los UIDs pendientes. `acquisition_family` sólo participa en el balance de folds y las auditorías: nunca entra a los predictores.


In [2]:
with RunLock(RUN_DIR / "prepare.lock"):
    prepared = prepare_experiment(EXPERIMENT, project_root=PROJECT_ROOT)

display(pd.DataFrame({
    "indicador": [
        "casos", "UID únicos", "familias base", "folds por trial",
        "mínimo COMPLETE por modelo", "folds finales", "entrenamientos CV mínimos de búsqueda",
    ],
    "valor": [
        len(prepared.cohort), prepared.cohort["uid"].nunique(),
        len(EXPERIMENT.model_families), EXPERIMENT.search.n_splits_search,
        EXPERIMENT.search.effective_completed_trials, EXPERIMENT.search.n_splits_final,
        len(EXPERIMENT.model_families) * EXPERIMENT.search.effective_completed_trials * EXPERIMENT.search.n_splits_search,
    ],
}))
display(prepared.cohort[[
    "uid", "is_pathologic", "intensity_l2_norm", "intensity_mean",
    "intensity_p90", "intensity_positive_voxels",
]].head())


,indicador,valor
0,casos,1362
1,UID únicos,1362
2,familias base,9
3,folds por trial,3
4,mínimo COMPLETE por modelo,10
5,folds finales,5
6,entrenamientos CV mínimos de búsqueda,270


,uid,is_pathologic,intensity_l2_norm,intensity_mean,intensity_p90,intensity_positive_voxels
0,01nouhtc,0,2.509983e+05,2024.381592,2787.359375,14108
1,0224wk0y,1,2.185329e+03,18.119324,22.587933,14098
2,049enulq,0,1.864536e+06,15156.818359,19407.101562,14140
3,04x0qzfh,0,1.561642e+03,12.894911,15.279227,14086
4,05xt54fn,0,3.554008e+05,2925.310791,3496.289307,14121


## 2. Nueve búsquedas base con Optuna

Cada estudio debe alcanzar al menos diez trials en estado `COMPLETE`. Un trial `PRUNED` no cuenta: se solicita otro automáticamente. Los modelos neuronales guardan `last.pt`, `last.prev.pt`, `best.pt`, historia por época y predicciones por fold. Los modelos sklearn persisten cada fold completo y Optuna conserva el estado en SQLite.

La ejecución puede ser larga: son como mínimo 270 evaluaciones de fold. Si se apaga el equipo, vuelve a ejecutar esta celda con el mismo `RUN_ID`.


In [3]:
with RunLock(RUN_DIR / "search.lock"):
    search_result = run_search_stage(prepared, EXPERIMENT, device=DEVICE)
display(search_result.summary)
assert (search_result.summary["completed_trials"] >= 10).all()


[I 2026-09-02 07:39:45,108] Using an existing study with name 'node10_dual_stream_multitask' instead of creating a new one.
[I 2026-09-02 07:39:45,134] Using an existing study with name 'node10_regional_hgb' instead of creating a new one.
[I 2026-09-02 07:39:45,152] Using an existing study with name 'node10_topology_hgb' instead of creating a new one.
[I 2026-09-02 07:39:45,162] Using an existing study with name 'node10_graph_elasticnet' instead of creating a new one.
[I 2026-09-02 07:39:45,185] Using an existing study with name 'node10_graph_random_forest' instead of creating a new one.
[I 2026-09-02 07:39:45,210] Using an existing study with name 'node10_graph_mlp' instead of creating a new one.
[I 2026-09-02 07:39:45,229] Using an existing study with name 'node10_graph_gcn' instead of creating a new one.
[I 2026-09-02 07:39:55,466] Trial 1 finished with value: 0.6660196582476298 and parameters: {'hidden_dim': 128, 'dropout': 0.24562844121246347, 'learning_rate': 3.462009887168136e-0

,study_name,family,completed_trials,total_trials,search_log_loss,fixed_epochs,candidate_id
0,node10_regional_hgb,regional_hgb,10,11,0.533516,NaN,473cd649f8ef9aa0
1,node10_graph_random_forest,graph_random_forest,10,45,0.551590,NaN,8ffd11a417562936
2,node10_topology_hgb,topology_hgb,10,10,0.554139,NaN,1456e74dd11d082b
3,node10_graph_mlp,graph_mlp,10,22,0.567252,10.0,f272865b491010ff
4,node10_graph_gcn,graph_gcn,10,12,0.603119,13.0,eb916479db2a93ed
5,node10_dual_stream_multitask,dual_stream_multitask,10,11,0.609528,11.0,1944b93a1f2dea5f
6,node10_subtype_mixture,subtype_mixture,10,11,0.685552,9.0,9589112f4faa7a7c
7,node10_diffusion_map,diffusion_map,10,15,0.692066,NaN,8ea47aa43b21ec24
8,node10_graph_elasticnet,graph_elasticnet,10,12,0.709481,NaN,0934fa9b01867d1d


## 3. CV5 congelado, calibración y stacking

El ganador de cada familia se reentrena en cinco folds congelados. Después se construye una matriz OOF común: una media igual preespecificada permanece elegible como modelo primario y un décimo estudio Optuna ajusta un meta-modelo Elastic Net sobre logits. Este stacking es exploratorio porque una estimación promocionable exigiría regenerar los expertos dentro de un segundo nivel estrictamente anidado. El meta-modelo también exige diez trials completos. La probabilidad reportada se cross-calibra sin usar el fold evaluado para ajustar su temperatura.

La evaluación guarda además intervalos bootstrap de log loss, peor familia de adquisición con soporte $n\geq10$ y matrices de correlación/diferencia entre expertos. La familia de adquisición se une sólo después de predecir, exclusivamente para auditoría.


In [4]:
with RunLock(RUN_DIR / "final.lock"):
    final_result = run_final_stage(prepared, EXPERIMENT, device=DEVICE)
display(final_result.metrics)
display(Markdown(
    f"**Primario elegible:** `{final_result.deployment_manifest['primary_family']}` · "
    f"**expertos base:** `{final_result.deployment_manifest['n_base_experts']}` · "
    f"**folds por experto:** `{final_result.deployment_manifest['n_fold_models_per_expert']}`"
))


[nodo10 dual] fold=0 epoch=1/11 val=0.6880
[nodo10 dual] fold=0 epoch=2/11 val=0.6667
[nodo10 dual] fold=0 epoch=3/11 val=0.6598
[nodo10 dual] fold=0 epoch=4/11 val=0.6765
[nodo10 dual] fold=0 epoch=5/11 val=0.6427
[nodo10 dual] fold=0 epoch=6/11 val=0.6192
[nodo10 dual] fold=0 epoch=7/11 val=0.6248
[nodo10 dual] fold=0 epoch=8/11 val=0.6068
[nodo10 dual] fold=0 epoch=9/11 val=0.6077
[nodo10 dual] fold=0 epoch=10/11 val=0.6110
[nodo10 dual] fold=0 epoch=11/11 val=0.6126
[nodo10 dual] fold=1 epoch=1/11 val=0.6851
[nodo10 dual] fold=1 epoch=2/11 val=0.6923
[nodo10 dual] fold=1 epoch=3/11 val=0.6685
[nodo10 dual] fold=1 epoch=4/11 val=0.6770
[nodo10 dual] fold=1 epoch=5/11 val=0.6894
[nodo10 dual] fold=1 epoch=6/11 val=0.6686
[nodo10 dual] fold=1 epoch=7/11 val=0.6713
[nodo10 dual] fold=1 epoch=8/11 val=0.6585
[nodo10 dual] fold=1 epoch=9/11 val=0.6858
[nodo10 dual] fold=1 epoch=10/11 val=0.6724
[nodo10 dual] fold=1 epoch=11/11 val=0.6720
[nodo10 dual] fold=2 epoch=1/11 val=0.6925
[nodo10

c:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\alana\OneDrive\Documentos\ChatGPT\DaT Scan\dat-parkinsons-challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid th

,family,candidate_id,raw_log_loss,raw_brier,raw_auc,raw_balanced_accuracy_0_5,raw_ece_10,calibrated_log_loss,calibrated_brier,calibrated_auc,calibrated_balanced_accuracy_0_5,calibrated_ece_10,final_temperature,bootstrap_draws,calibrated_log_loss_ci_low,calibrated_log_loss_ci_high,worst_supported_family_log_loss,promotion_eligible
0,hybrid_stacking,node10_hybrid_stacking,0.510015,0.168984,0.826221,0.749244,0.017545,0.510131,0.169032,0.825920,0.749244,0.015099,1.031788,1000,0.484873,0.537924,0.848410,False
1,regional_hgb,473cd649f8ef9aa0,0.519093,0.172344,0.819462,0.759291,0.041084,0.519639,0.172479,0.818881,0.759291,0.042757,1.047729,1000,0.493401,0.547555,0.845484,True
2,topology_hgb,1456e74dd11d082b,0.536630,0.178871,0.807081,0.729258,0.042796,0.533735,0.178106,0.806992,0.729258,0.031652,1.171300,1000,0.507392,0.560387,0.816383,True
3,graph_random_forest,8ffd11a417562936,0.555762,0.186728,0.791038,0.724952,0.044480,0.555067,0.186246,0.789981,0.724952,0.028873,0.872216,1000,0.528225,0.581097,0.812426,True
4,graph_mlp,f272865b491010ff,0.576831,0.195171,0.769778,0.707500,0.040666,0.577746,0.195646,0.768914,0.707500,0.039466,1.061773,1000,0.555513,0.602069,0.799246,True
5,graph_gcn,eb916479db2a93ed,0.613585,0.211356,0.727685,0.690388,0.046725,0.613923,0.211627,0.726490,0.690388,0.043180,1.126259,1000,0.594190,0.634814,0.880531,True
6,dual_stream_multitask,1944b93a1f2dea5f,0.635926,0.220516,0.708366,0.653992,0.065987,0.626595,0.217854,0.704137,0.653992,0.030530,1.477475,1000,0.607536,0.645704,0.795982,True
7,subtype_mixture,9589112f4faa7a7c,0.689726,0.248346,0.526205,0.506533,0.052301,0.689692,0.248331,0.526205,0.506533,0.053103,0.979897,1000,0.687611,0.691731,0.694575,True
8,diffusion_map,8ea47aa43b21ec24,0.692747,0.249823,0.501874,0.501473,0.048059,0.692747,0.249823,0.501874,0.501473,0.048058,0.998173,1000,0.691773,0.693188,0.693657,True
9,graph_elasticnet,0934fa9b01867d1d,0.712822,0.251628,0.576356,0.553659,0.068724,0.693546,0.250190,0.576345,0.553659,0.065604,20.000000,1000,0.692663,0.694512,0.693615,True


**Primario elegible:** `regional_hgb` · **expertos base:** `9` · **folds por experto:** `5`

## 4. Lectura científica

No se promueve automáticamente el modelo más complejo. La comparación central es:

1. ¿La rama dual supera a las ramas previas en log loss CV5?
2. ¿GCN supera a Elastic Net, Random Forest y MLP usando exactamente los mismos nodos?
3. ¿Topología o Diffusion Maps reducen errores distintos?
4. ¿La media preespecificada mejora sin empeorar calibración ni peor familia?
5. ¿El stacking exploratorio justifica pagar el costo de una validación de segundo nivel anidada?

Las regiones son proxies geométricos reproducibles, no segmentaciones clínicas. Un resultado prometedor debe repetirse con semillas, bootstrap de diferencias y auditoría por familia antes de preparar un submission.
